## EV Driver POI Visit Behavior During Charging Sessions 
Author: Callie Clark

In [ ]:
!pip install pydeck -q -q
# ! pip install statsmodels
# !pip install seaborn
exec(open('../EV00_settings.py').read())
import matplotlib.pyplot as plt

In [ ]:


def bootstrap_category_ci(df, n_bootstrap=1000,seed=42):
    categories = ['Beauty', 'Entertainment', 'Errands', 'Fitness', 
                  'Grocery', 'Pharmacies', 'Restaurants', 'Retail']

    np.random.seed(seed)
        
    # Pre-extract the relevant columns as numpy arrays for speed
    category_data = df[categories].values
    n_samples = len(df)
    print(n_samples)
    
    # Pre-allocate results array
    results = np.zeros((n_bootstrap, len(categories)))
    
    for i in range(n_bootstrap):
        # Generate bootstrap indices
        bootstrap_indices = np.random.choice(n_samples, size=n_samples, replace=True)
        
        # Calculate means for this bootstrap sample
        bootstrap_sample = category_data[bootstrap_indices]
        results[i] = bootstrap_sample.mean(axis=0)
    # Compute statistics
    mean_values = results.mean(axis=0)
    lower_ci = np.percentile(results, 2.5, axis=0)
    upper_ci = np.percentile(results, 97.5, axis=0)
    
    print(mean_values)
    # Convert back to pandas Series with proper index
    mean_values = pd.Series(mean_values, index=categories)
    lower_ci = pd.Series(lower_ci, index=categories)
    upper_ci = pd.Series(upper_ci, index=categories)
    
    return mean_values, lower_ci, upper_ci



In [ ]:
current_city #denver, 'SF', 'Boston', ''

## Generating DFS

In [ ]:


evcs_visits_all=pd.DataFrame()      
# convert inputs to datetime
start_dt = dt.strptime(start_date, '%Y%m%d')
end_dt = dt.strptime(end_date, '%Y%m%d')


current_date = start_dt
while current_date <= end_dt:
    date_str = str(current_date.strftime('%Y%m%d'))
    print(f"-------------------------------------------------\nProcessing date: {date_str}")
    

    evcs_stop_df=pd.read_csv(f'{current_session_path}evcs_session_stops/model/driver_EVCS_behavior_{current_city}_{date_str}.csv',index_col=0)
    evcs_stop_df=evcs_stop_df[evcs_stop_df['classification_type']!='RECURRING_AREA']

    #evcs_visits=evcs_stop_df.dropna(subset='placekey')
    evcs_visits_all=pd.concat([evcs_visits_all,evcs_stop_df])
    

   
    
    current_date = current_date + timedelta(days = 1)
print(len(evcs_visits_all))



print('Aggregated, adding one-hot encoding')
df_encoded = pd.get_dummies(evcs_visits_all['category'], prefix='', prefix_sep='')
evcs_stops=pd.concat([evcs_visits_all,df_encoded],axis=1)
evcs_stops.reset_index().drop(columns=['index']).to_csv(current_session_path+f'ev_driver_info/all_visits_{current_city}.csv')
evcs_poi_stops=evcs_stops.dropna(subset='placekey') #subset to stops with a POI visit
print(len(evcs_poi_stops))
evcs_poi_stops.reset_index().drop(columns=['index']).to_csv(current_session_path+f'ev_driver_info/all_poi_visits_{current_city}.csv')
print('Done')

In [ ]:
#agg evcs_id to station 
nonevcs_visits_all=pd.DataFrame()    
# convert inputs to datetime
start_dt = dt.strptime(start_date, '%Y%m%d')
end_dt = dt.strptime(end_date, '%Y%m%d')

current_date = start_dt
while current_date <= end_dt:
    date_str = str(current_date.strftime('%Y%m%d'))
    print(f"-------------------------------------------------\nProcessing date: {date_str}")
    nonevcs_stops=pd.read_csv(current_session_path+f'all_driver_info/poi_visits_{current_city}_{date_str}.csv',index_col=0)
    nonevcs_poi_stops=nonevcs_stops.dropna(subset='placekey')
    nonevcs_visits_all=pd.concat([nonevcs_visits_all,nonevcs_poi_stops])
    current_date = current_date + timedelta(days = 1)

print('Aggregated, adding one-hot encoding')
nonevcs_visits_all.to_csv(current_session_path+f'all_driver_info/all_poi_visits_{current_city}_LB.csv.gz',compression='gzip')
df_encoded = pd.get_dummies(nonevcs_visits_all['category'], prefix='', prefix_sep='')
nonevcs_stops_=pd.concat([nonevcs_visits_all,df_encoded],axis=1)
# nonevcs_stops_.reset_index().drop(columns=['index']).to_csv(current_session_path+f'all_driver_info/all_visits.csv')
del nonevcs_visits_all,df_encoded
#del nonevcs_stops

nonevcs_stops_.reset_index().drop(columns=['index']).to_csv(current_session_path+f'all_driver_info/all_poi_visits_{current_city}.csv.gz',compression='gzip')
print('Done')

In [ ]:
#filter so we are comparing the same EVCS stations
poi_evcs_stops_df=pd.read_csv(current_session_path+f'ev_driver_info/all_poi_visits_{current_city}.csv')
poi_nonevcs_stops_df=pd.read_csv(current_session_path+f'all_driver_info/all_poi_visits_{current_city}.csv.gz',index_col=0)
print(len(poi_evcs_stops_df))
print(len(poi_nonevcs_stops_df))
poi_nonevcs_stops_df_filtered=poi_nonevcs_stops_df[poi_nonevcs_stops_df['evcs_id'].isin(poi_evcs_stops_df['evcs_id'].unique())]
print(len(poi_nonevcs_stops_df_filtered))
poi_evcs_stops_df_filtered=poi_evcs_stops_df[poi_evcs_stops_df['evcs_id'].isin(poi_nonevcs_stops_df_filtered['evcs_id'].unique())]
print(len(poi_evcs_stops_df_filtered))
poi_nonevcs_stops_df_filtered.to_csv(current_session_path+f'all_driver_info/all_poi_visits_{current_city}_subset.csv.gz',compression='gzip')
poi_evcs_stops_df_filtered.to_csv(current_session_path+f'ev_driver_info/all_poi_visits_{current_city}_subset.csv.gz',compression='gzip')
print('done')

## Aggregating DFs to Person-level

In [ ]:
# these csvs look at the same subset of evcs 

poi_evcs_stops=pd.read_csv(current_session_path+f'ev_driver_info/all_poi_visits_{current_city}_subset.csv.gz')
poi_nonevcs_stops=pd.read_csv(current_session_path+f'all_driver_info/all_poi_visits_{current_city}_subset.csv.gz',index_col=0)
poi_evcs_stops.head()

In [ ]:
cols = ['Pharmacies', 'Restaurants', 'Retail', 'Grocery', 'Beauty', 'Errands', 'Fitness','Entertainment']

ev_csv =current_session_path+ 'ev_driver_info/'
ev_user_1 = pd.read_csv(ev_csv+f'ev_driver_info_model_{current_city}_2022_4_zip_demographics.csv',index_col=0)
ev_user_2 = pd.read_csv(ev_csv+f'ev_driver_info_model_{current_city}_2022_5_zip_demographics.csv',index_col=0)
ev_user_3 = pd.read_csv(ev_csv+f'ev_driver_info_model_{current_city}_2022_3_zip_demographics.csv',index_col=0)
ev_user_= pd.concat([ev_user_1,ev_user_2,ev_user_3])
ev_user_info=ev_user_.drop_duplicates(subset='cuebiq_id')

#this is counting the number of charging sessions and poi visits by category for each user
ev_df1=poi_evcs_stops[cols+['session_id','cuebiq_id']].groupby('cuebiq_id').agg({'Beauty':'sum', 'Entertainment':'sum', 'Errands':'sum', 'Fitness':'sum', 
                  'Grocery':'sum', 'Pharmacies':'sum', 'Restaurants':'sum','Retail':'sum','session_id':'count'})

#Below we divide number of POI visits by the number of charging sessions 
ev_df2=ev_df1.copy()
for col in cols:
    print(col)
    ev_df2[col]=ev_df2[col]/ev_df2['session_id']
ev_df2=ev_df2.reset_index()
#the output is each users probability of visiting a type of POI given they visits a POI 
ev_df2.to_csv(current_session_path+f'ev_driver_info/person_poi_visits_{current_city}.csv')
ev_df2

In [ ]:
#user info


user_csv =current_session_path+ '/all_driver_info/'
user_1 = pd.read_csv(user_csv+f'all_driver_info_{current_city}_2022_4_zip_demographics.csv',index_col=0)
user_2 = pd.read_csv(user_csv+f'all_driver_info_{current_city}_2022_5_zip_demographics.csv',index_col=0)
user_3 = pd.read_csv(user_csv+f'all_driver_info_{current_city}_2022_3_zip_demographics.csv',index_col=0)
user_= pd.concat([user_1,user_2,user_3])
user_info=user_.drop_duplicates(subset='cuebiq_id')

poi_nonevcs_stops['date']=poi_nonevcs_stops.stop_zoned_datetime.str[:10]
poi_nonevcs_stops['session_id']=poi_nonevcs_stops['date']+'_'+poi_nonevcs_stops['evcs_id'].astype(int).astype(str)


# all_df1=poi_nonevcs_stops[cols+['session_id','cuebiq_id']].groupby('cuebiq_id').agg({'Beauty':'sum', 'Entertainment':'sum', 'Errands':'sum', 'Fitness':'sum', 
#                   'Grocery':'sum', 'Pharmacies':'sum', 'Restaurants':'sum','Retail':'sum','session_id':'count'})

all_df1=poi_nonevcs_stops[cols+['session_id','cuebiq_id']].groupby('cuebiq_id').agg({'Beauty':'sum', 'Entertainment':'sum', 'Errands':'sum', 'Fitness':'sum', 
                  'Grocery':'sum', 'Pharmacies':'sum', 'Restaurants':'sum','Retail':'sum','session_id':'count'})

all_df2=all_df1.copy()
for col in cols:
    print(col)
    all_df2[col]=all_df2[col]/all_df2['session_id']
all_df2=all_df2.reset_index()
#the output is each users probability of visitting a type of POI given they visits a POI 
all_df2.to_csv(current_session_path+f'all_driver_info/person_poi_visits_{current_city}.csv')
all_df2

In [ ]:
poi_prox_count=pd.read_csv(root+f'data/POI_data/{current_city}_evcs_poi_counts_{max_distance_POI_to_EVCS+50}m.csv')
poi_prox_count=poi_prox_count[poi_prox_count.ID.isin(poi_evcs_stops['evcs_id'])].copy()
poi_prox_count[cols] = poi_prox_count[cols].div(poi_prox_count[cols].sum(axis=1), axis=0)
poi_prox_count=poi_prox_count.dropna() 
poi_prox_count_w=poi_prox_count.copy()
#poi_prox_count_w=poi_evcs_stops[['evcs_id']].merge(poi_prox_count,left_on='evcs_id',right_on='ID',how='inner')

poi_prox_count_w=poi_prox_count_w.drop_duplicates()
poi_prox_count_w.to_csv(root+f'data/POI_data/{current_city}_poi_prox_count_{max_distance_POI_to_EVCS+50}m.csv')

## Analyze DFs

In [ ]:
# poi_evcs_stops_df=pd.read_csv(current_session_path+f'ev_driver_info/all_poi_visits_{current_city}_model.csv')
# poi_nonevcs_stops_df=pd.read_csv(current_session_path+f'all_driver_info/all_poi_visits_{current_city}.csv.gz',index_col=0)


In [ ]:
cols = [ 'Pharmacies', 'Restaurants', 'Retail', 'Grocery', 'Beauty', 'Errands', 'Fitness','Entertainment']

# poi_evcs_stops=pd.read_csv(current_session_path+f'ev_driver_info/all_poi_visits_{current_city}_model.csv')
# poi_nonevcs_stops=pd.read_csv(current_session_path+f'all_driver_info/all_poi_visits_{current_city}.csv.gz',index_col=0)

poi_evcs_stops=pd.read_csv(current_session_path+f'ev_driver_info/person_poi_visits_{current_city}.csv')
poi_nonevcs_stops=pd.read_csv(current_session_path+f'all_driver_info/person_poi_visits_{current_city}.csv',index_col=0)

poi_prox_count_w=pd.read_csv(root+f'data/POI_data/{current_city}_poi_prox_count_{max_distance_POI_to_EVCS+50}m.csv')


# Plot Prob by category and group

In [ ]:
#mean_prox_bool, ci_low_prox_bool, ci_high_prox_bool = bootstrap_category_ci(poi_prox_data_w)
mean_prox_count, ci_low_prox_count, ci_high_prox_count = bootstrap_category_ci(poi_prox_count_w)
mean_evcs, ci_low_evcs, ci_high_evcs = bootstrap_category_ci(poi_evcs_stops)
mean_all, ci_low_all, ci_high_all = bootstrap_category_ci(poi_nonevcs_stops)

summary_df = pd.DataFrame({
    'Category':mean_prox_count.index.to_list()+mean_evcs.index.to_list()+mean_all.index.to_list(),
    'Mean': pd.concat([mean_prox_count, mean_evcs, mean_all], axis=0).values,
    'CI_Low': pd.concat([ci_low_prox_count, ci_low_evcs, ci_low_all], axis=0).values,
    'CI_High': pd.concat([ ci_high_prox_count,ci_high_evcs, ci_high_all], axis=0).values,
    'Group':['Spatial (count)'] * 8 + ['EVCS Subset'] * 8 + ['All Visits'] * 8
})
summary_df.to_csv(current_session_path+f'poi_visit_info/{current_city}_POI_visit_cat.csv')

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Setup
categories = summary_df['Category'].unique()
groups = ['Spatial (count)', 'All Visits', 'EVCS Subset']

# Map old group names to new display names
group_name_map = {
    'Spatial (count)': 'Accessibility',
    'All Visits': 'Popularity',
    'EVCS Subset': 'EV Drivers'
}

# Improved color palette (ColorBrewer Set2)
group_colors = {
    'Spatial (count)':'#fc8d62',  # soft green
    'All Visits':  '#8da0cb',       # soft orange
    'EVCS Subset': '#66c2a5'    # soft blue
}

n_categories = len(categories)
n_groups = len(groups)
bar_height = 0.15
group_spacing = bar_height * n_groups + 0.1

# Compute y positions for each group within each category
y_ticks = []
y_tick_labels = []
bar_positions = []

for i, cat in enumerate(categories):
    base_y = i * group_spacing
    y_ticks.append(base_y + bar_height * (n_groups - 1) / 2)
    y_tick_labels.append(cat)
    for j, group in enumerate(groups):
        y = base_y + j * bar_height
        bar_positions.append((cat, group, y))

# Plot
fig, ax = plt.subplots(figsize=(10,8))

for cat, group, y in bar_positions:
    row = summary_df[(summary_df['Category'] == cat) & (summary_df['Group'] == group)].iloc[0]
    mean = row['Mean']
    ci_low = row['CI_Low']
    ci_high = row['CI_High']
    color = group_colors[group]

    # Draw bar
    ax.barh(
        y=y,
        width=mean,
        height=bar_height * 0.9,
        color=color,
        edgecolor='black',
        alpha=0.7,
        label=group_name_map[group] if (cat == categories[0]) else ""
    )

    # Draw CI line with caps
    ax.errorbar(
        x=mean,
        y=y,
        xerr=[[mean - ci_low], [ci_high - mean]],
        fmt='none',
        ecolor='black',
        capsize=5,
        linewidth=1.5,
        zorder=3
    )

# Formatting
ax.set_yticks(y_ticks)
ax.set_yticklabels(y_tick_labels, fontsize=18)
ax.set_xlabel(f'{current_city}: Probability of Visiting POI', fontsize=18)
ax.set_ylabel('Top Category', fontsize=18)
ax.set_title(f'{current_city}: Probabilities of Visiting a POI with 95% CI by Group', fontsize=15)
ax.legend(title='Group', bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=11)
ax.axvline(0, color='gray', linestyle='--', linewidth=1)
ax.tick_params(axis='x', labelsize=18) 

# Optional: remove spines for a cleaner look
for spine in ['top', 'right', 'left']:
    ax.spines[spine].set_visible(False)

plt.tight_layout()


plt.savefig(os.path.join(f"../../figures/{current_city}_poi_visit_prob.png"), dpi=300)

plt.show()

